In [11]:
import gzip
import struct
import time
import urllib.request
from pathlib import Path

import numpy as np

np.random.seed(42)
print("NumPy version:", np.__version__)

NumPy version: 2.4.6


In [12]:
class Dense:
    def __init__(self, input_size, output_size, rng):
        # He initialization for layers followed by ReLU
        self.W = (
            rng.standard_normal((input_size, output_size)).astype(np.float32)
            * np.sqrt(2.0 / input_size)
        )
        self.b = np.zeros((1, output_size), dtype=np.float32)
        self.x = None

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, grad_output):
        grad_W = self.x.T @ grad_output
        grad_b = np.sum(grad_output, axis=0, keepdims=True)
        grad_input = grad_output @ self.W.T
        return grad_input, {"W": grad_W, "b": grad_b}

    def update(self, grads, learning_rate):
        self.W -= learning_rate * grads["W"]
        self.b -= learning_rate * grads["b"]

In [13]:
# CELL 3 — Activation functions, Softmax and Cross-Entropy
def relu(z):
    return np.maximum(0.0, z)

def relu_backward(grad_output, z):
    return grad_output * (z > 0)

def softmax(logits):
    # Subtract maximum for numerical stability
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=1, keepdims=True)

def cross_entropy_from_logits(logits, y):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    log_sum_exp = np.log(np.sum(np.exp(shifted), axis=1))
    correct_logits = shifted[np.arange(len(y)), y]
    return float(np.mean(log_sum_exp - correct_logits))


In [14]:
# CELL 4 — Multilayer Perceptron
class MLP:
    def __init__(self, input_size=784, hidden1=128, hidden2=64,
                 output_size=10, seed=42):
        rng = np.random.default_rng(seed)
        self.fc1 = Dense(input_size, hidden1, rng)
        self.fc2 = Dense(hidden1, hidden2, rng)
        self.fc3 = Dense(hidden2, output_size, rng)

    def forward(self, x):
        self.z1 = self.fc1.forward(x)
        self.a1 = relu(self.z1)
        self.z2 = self.fc2.forward(self.a1)
        self.a2 = relu(self.z2)
        self.logits = self.fc3.forward(self.a2)
        self.probs = softmax(self.logits)
        return self.probs

    def backward(self, y):
        n = len(y)
        # Derivative of Softmax + cross-entropy
        grad_logits = self.probs.copy()
        grad_logits[np.arange(n), y] -= 1.0
        grad_logits /= n

        grad_a2, grads3 = self.fc3.backward(grad_logits)
        grad_z2 = relu_backward(grad_a2, self.z2)

        grad_a1, grads2 = self.fc2.backward(grad_z2)
        grad_z1 = relu_backward(grad_a1, self.z1)

        _, grads1 = self.fc1.backward(grad_z1)
        return [grads1, grads2, grads3]

    def update(self, all_grads, learning_rate):
        for layer, grads in zip((self.fc1, self.fc2, self.fc3), all_grads):
            layer.update(grads, learning_rate)

    def predict(self, x):
        return np.argmax(self.forward(x), axis=1)

    def evaluate(self, x, y, batch_size=512):
        correct = 0
        total_loss = 0.0
        seen = 0

        for start in range(0, len(x), batch_size):
            xb = x[start:start + batch_size]
            yb = y[start:start + batch_size]
            probs = self.forward(xb)
            correct += int(np.sum(np.argmax(probs, axis=1) == yb))
            total_loss += cross_entropy_from_logits(self.logits, yb) * len(xb)
            seen += len(xb)

        return total_loss / seen, correct / seen

    def train_epoch(self, x, y, batch_size=128, learning_rate=0.1, seed=None):
        rng = np.random.default_rng(seed)
        order = rng.permutation(len(x))
        total_loss = 0.0

        for start in range(0, len(x), batch_size):
            indices = order[start:start + batch_size]
            xb, yb = x[indices], y[indices]

            self.forward(xb)
            total_loss += cross_entropy_from_logits(self.logits, yb) * len(xb)
            grads = self.backward(yb)
            self.update(grads, learning_rate)

        return total_loss / len(x)


In [15]:
# CELL 5 — Download and read the official MNIST IDX files
def load_mnist(data_dir="data/mnist"):
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    urls = {
        "train_images": "https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz",
        "train_labels": "https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz",
        "test_images": "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-images-idx3-ubyte.gz",
        "test_labels": "https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz",
    }

    arrays = {}
    for name, url in urls.items():
        file_path = data_dir / f"{name}.gz"

        if not file_path.exists():
            print("Downloading:", name)
            urllib.request.urlretrieve(url, file_path)

        with gzip.open(file_path, "rb") as f:
            raw = f.read()

        if "images" in name:
            magic, count, rows, cols = struct.unpack(">IIII", raw[:16])
            if magic != 2051:
                raise ValueError(f"Invalid image IDX file: {name}")
            arrays[name] = np.frombuffer(
                raw, dtype=np.uint8, offset=16
            ).reshape(count, rows * cols).copy()
        else:
            magic, count = struct.unpack(">II", raw[:8])
            if magic != 2049:
                raise ValueError(f"Invalid label IDX file: {name}")
            arrays[name] = np.frombuffer(
                raw, dtype=np.uint8, offset=8
            ).copy()

    # Flattened 28 x 28 images; normalize pixels to [0, 1]
    x_train = arrays["train_images"].astype(np.float32) / 255.0
    y_train = arrays["train_labels"].astype(np.int64)
    x_test = arrays["test_images"].astype(np.float32) / 255.0
    y_test = arrays["test_labels"].astype(np.int64)

    return x_train, y_train, x_test, y_test

x_train, y_train, x_test, y_test = load_mnist()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)


Training images: (60000, 784)
Training labels: (60000,)
Testing images: (10000, 784)
Testing labels: (10000,)


In [16]:
# CELL 6 — Verify the forward pass
sample = np.random.default_rng(1).random((4, 784), dtype=np.float32)
test_model = MLP()
sample_probs = test_model.forward(sample)

assert sample_probs.shape == (4, 10)
assert np.allclose(sample_probs.sum(axis=1), 1.0, atol=1e-6)

print("Forward-pass tests passed.")
print("Output shape:", sample_probs.shape)
print("Probability sums:", sample_probs.sum(axis=1))


Forward-pass tests passed.
Output shape: (4, 10)
Probability sums: [1. 1. 1. 1.]


In [17]:
# CELL 7 — Numerical gradient checking
def gradient_check(seed=7, epsilon=1e-3, checks=8):
    rng = np.random.default_rng(seed)
    model = MLP(input_size=4, hidden1=5, hidden2=4,
                output_size=3, seed=seed)

    x = rng.normal(size=(3, 4)).astype(np.float32)
    y = np.array([0, 2, 1], dtype=np.int64)

    model.forward(x)
    analytic_grads = model.backward(y)

    params = [
        (model.fc1, "W", analytic_grads[0]["W"]),
        (model.fc1, "b", analytic_grads[0]["b"]),
        (model.fc2, "W", analytic_grads[1]["W"]),
        (model.fc2, "b", analytic_grads[1]["b"]),
        (model.fc3, "W", analytic_grads[2]["W"]),
        (model.fc3, "b", analytic_grads[2]["b"]),
    ]

    errors = []
    for layer, param_name, grad in params:
        param = getattr(layer, param_name)

        for _ in range(checks):
            idx = tuple(rng.integers(0, size) for size in param.shape)
            original = float(param[idx])

            param[idx] = original + epsilon
            model.forward(x)
            loss_plus = cross_entropy_from_logits(model.logits, y)

            param[idx] = original - epsilon
            model.forward(x)
            loss_minus = cross_entropy_from_logits(model.logits, y)

            param[idx] = original
            numerical = (loss_plus - loss_minus) / (2 * epsilon)
            analytical = float(grad[idx])

            relative_error = abs(numerical - analytical) / max(
                1e-7, abs(numerical) + abs(analytical)
            )
            errors.append(relative_error)

    return max(errors), errors

max_error, gradient_errors = gradient_check()
print(f"Maximum relative gradient error: {max_error:.3e}")
print("Gradient check completed.")


Maximum relative gradient error: 1.749e-06
Gradient check completed.


In [18]:
# CELL 8 — Train the NumPy MLP on MNIST
def run_mnist(epochs=5, batch_size=128, learning_rate=0.1):
    model = MLP()
    history = []

    for epoch in range(epochs):
        start_time = time.time()

        train_loss = model.train_epoch(
            x_train, y_train,
            batch_size=batch_size,
            learning_rate=learning_rate,
            seed=epoch
        )

        test_loss, test_accuracy = model.evaluate(x_test, y_test)
        elapsed = time.time() - start_time

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "test_loss": test_loss,
            "test_accuracy": test_accuracy,
            "seconds": elapsed,
        })

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"test_loss={test_loss:.4f} | "
            f"test_accuracy={test_accuracy:.2%} | "
            f"time={elapsed:.1f}s"
        )

    return model, history

# Start with 5 epochs. Training time depends on the Colab runtime.
start_time = time.time()
numpy_model, numpy_history = run_mnist(
    epochs=5,
    batch_size=128,
    learning_rate=0.1
)
numpy_runtime = time.time() - start_time

final_test_loss, final_test_accuracy = numpy_model.evaluate(x_test, y_test)
print("\nNUMPY MLP FINAL RESULTS")
print("Test loss:", round(final_test_loss, 4))
print("Test accuracy:", f"{final_test_accuracy:.2%}")
print("Total runtime (seconds):", round(numpy_runtime, 2))


Epoch 1/5 | train_loss=0.4155 | test_loss=0.2250 | test_accuracy=93.58% | time=11.0s
Epoch 2/5 | train_loss=0.2033 | test_loss=0.1616 | test_accuracy=95.33% | time=2.6s
Epoch 3/5 | train_loss=0.1519 | test_loss=0.1419 | test_accuracy=95.56% | time=5.9s
Epoch 4/5 | train_loss=0.1227 | test_loss=0.1192 | test_accuracy=96.31% | time=3.1s
Epoch 5/5 | train_loss=0.1027 | test_loss=0.1035 | test_accuracy=96.86% | time=3.9s

NUMPY MLP FINAL RESULTS
Test loss: 0.1035
Test accuracy: 96.86%
Total runtime (seconds): 26.62
